## Concept 7 — Multi-Agent (Router Pattern)

### What is it?
A router that sends each query to a **specialized `create_agent` sub-agent** tuned for that
task type. Each sub-agent has a focused system prompt and only the tools it needs.
Better accuracy than one general-purpose agent handling everything.

### vs Single Agent (Notebooks 1–6)
```
Single agent (all tools, generic prompt):
  One system prompt handles math, docs, and weather.
  May call the wrong tool on ambiguous queries.

Multi-agent (each specialized):
  Math Agent:    tools=[calculate],   'You are a math specialist'
  Docs Agent:    tools=[search_docs], 'You are a docs specialist'
  Weather Agent: tools=[get_weather], 'You report weather'
  Each tuned for one job → higher accuracy.
```

### Pipeline
```
Query → Router (LLM) → 'math' / 'docs' / 'weather'
  → math_agent    (create_agent, only calculate)
  → docs_agent    (create_agent, only search_docs)
  → weather_agent (create_agent, only get_weather)
  → Final Answer
```

### Limitation (what Concept 8 fixes)
Still reactive — no upfront planning for multi-step tasks.
→ Concept 8 generates a full plan before executing any step.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, calculate, search_docs, get_weather, TEST_QUERIES, run_queries
from langchain.agents import create_agent

llm = get_llm()
print('Creating specialized sub-agents...')

### Step 1 — Create Specialized Sub-Agents
Each `create_agent` gets a narrow system prompt and only its domain tool.
Compare this with your existing `7MultiAgent.ipynb` which uses the same pattern —
here we use the common use case tools from AgentUtils.

In [ ]:
math_agent = create_agent(
    model=llm,
    tools=[calculate],
    system_prompt=(
        'You are a math specialist. '
        'Use the calculate tool for ALL arithmetic — never compute in your head. '
        'Give concise answers like: "144 / 12 = 12.0"'
    ),
)

docs_agent = create_agent(
    model=llm,
    tools=[search_docs],
    system_prompt=(
        'You are a documentation specialist. '
        'Always call search_docs to find answers. '
        'Give clear explanations based on search results.'
    ),
)

weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        'You report weather information. '
        'Always call get_weather to look up conditions. '
        'Report temperature and conditions clearly.'
    ),
)

print('Agents created: math_agent, docs_agent, weather_agent')

### Step 2 — LLM Router
A separate LLM call classifies the query. No tool needed — pure LLM classification.

In [ ]:
def llm_route(query: str) -> str:
    prompt = (
        f'Classify this query into exactly one category:\n'
        f'  math    — arithmetic, numbers, calculation\n'
        f'  docs    — LangChain, agents, RAG, AI documentation\n'
        f'  weather — weather, temperature, forecast\n'
        f'Query: {query}\n'
        f'Reply with exactly one word: math, docs, or weather'
    )
    decision = llm.invoke(prompt).content.strip().lower()
    if 'math'    in decision: return 'math'
    if 'weather' in decision: return 'weather'
    return 'docs'

print('Router test:')
for q in TEST_QUERIES:
    print(f'  "{q[:55]}" → {llm_route(q)}')

### Step 3 — Wire Router + Agents

In [ ]:
agent_registry = {
    'math':    math_agent,
    'docs':    docs_agent,
    'weather': weather_agent,
}

def multi_agent(query: str) -> str:
    category = llm_route(query)
    print(f'  [Router → {category}_agent]')
    result = agent_registry[category].invoke({'messages': [{'role': 'user', 'content': query}]})
    return result['messages'][-1].content

### Step 4 — Keyword Router (No LLM Cost)
For well-defined categories, keyword matching is instant and costs nothing.

In [ ]:
import re

MATH_KW    = {'calculate','compute','multiply','divide','add','subtract','plus','minus','times'}
WEATHER_KW = {'weather','temperature','forecast','sunny','rainy','cloudy'}

def keyword_route(query: str) -> str:
    q = query.lower()
    if any(k in q for k in MATH_KW) or re.search(r'\d+\s*[*/+-]\s*\d+', q): return 'math'
    if any(k in q for k in WEATHER_KW):                                        return 'weather'
    return 'docs'

print('Keyword router (instant, no LLM API call):')
for q in TEST_QUERIES:
    print(f'  "{q[:55]}" → {keyword_route(q)}')

### Test — Standard Queries

In [ ]:
run_queries(multi_agent, label='Concept 7 — Multi-Agent Router')